# Creating a Simple Agent with Tracing

In [ ]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

In [3]:
from agents import Agent, Runner, function_tool, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a Math Tutor Agent

In [4]:
@function_tool
def add_numbers(first_number: float, second_number: float) -> float:
    """Return the sum of two numbers.

    Args:
        first_number: The first number to add.
        second_number: The second number to add.
    """
    return first_number + second_number


math_tutor = Agent(
    name="Math Tutor",
    instructions="""
    You are a patient, encouraging math tutor. Help the student understand the
    reasoning, not just get an answer.

    First identify the topic and likely level from the question. For a problem,
    explain the key idea and guide the student through the solution one step at
    a time. Prefer a useful hint or a question before revealing the full answer.
    If the student asks for a complete solution, show clear steps and verify the
    result. Explain mistakes kindly and ask a short check-for-understanding
    question when useful.

    Use the add_numbers tool when asked to add two numbers. Explain the result
    in the context of the student's question.
    Use correct mathematical notation. State any assumptions, and do not claim
    certainty when a problem is ambiguous. Keep explanations focused and adapt
    the detail to the student's level.
    """,
    tools=[add_numbers],
)

Let's execute the Agent:

In [ ]:
with trace("Math Tutor Agent"):
    result = await Runner.run(
        math_tutor,
        "Can you give me a hint for solving 2x + 5 = 17?",
    )

print(result)

Streaming the answer to the screen, token by token

In [ ]:
response_stream = Runner.run_streamed(
    math_tutor,
    "Can you give me a hint for solving 2x + 5 = 17?",
)

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

_Good Job!_